# 02. Multi-Cohort Dataset Audit and Patient Leakage Verification

This notebook audits patient ID partitions across cohorts to mathematically guarantee:
$$\text{Train} \cap \text{Val} = \emptyset, \quad \text{Train} \cap \text{Test} = \emptyset, \quad \text{Val} \cap \text{Test} = \emptyset, \quad \text{Source} \cap \text{Target} = \emptyset$$


In [ ]:
import pandas as pd
from drdg.data.splits import audit_patient_leakage, split_patient_stratified

# Construct sample multi-eye patient pool
data = []
for i in range(100):
    pid = f"demo::{i:03d}"
    data.append({"image_id": f"img_{i}_left", "patient_id": pid, "dr_grade": i % 5, "eye": "left"})
    data.append({"image_id": f"img_{i}_right", "patient_id": pid, "dr_grade": i % 5, "eye": "right"})
df = pd.DataFrame(data)

split = split_patient_stratified(df, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, seed=42)
audit = audit_patient_leakage(split.train_df, split.val_df, split.test_df)

print(f"Zero leakage verified: {not audit['has_leakage']}")
print(f"Train patients: {audit['n_train_patients']}, Val patients: {audit['n_val_patients']}, Test patients: {audit['n_test_patients']}")
